In [1]:
import numpy as np

# Read text

In [2]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:
print(f'Characters: {len(text)}')

Characters: 1115393


In [4]:
print(f'{text[:100]}')

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f'All characters: {"".join(chars)}')
print(f'Vocab size: {vocab_size}')

All characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocab size: 65


# Character level tokenizer

In [6]:
encoded_dict = {chars[i]: i for i in range(vocab_size)}
decoded_dict = {i: chars[i] for i in range(vocab_size)}

def encode(s: str) -> list[int]:
    return [encoded_dict[c] for c in s]

def decode(s: list[int]) -> str:
    return "".join([decoded_dict[c] for c in s])

print(encode("test string"))
print(decode(encode("test string")))

[58, 43, 57, 58, 1, 57, 58, 56, 47, 52, 45]
test string


In [7]:
data = encode(text)
data = np.array(data, dtype=np.longfloat).reshape(-1, 1)

In [8]:
n = int(0.9*len(data))
X_train, X_test = data[:n], data[n:]
#print(f'n: {n}\nX_train: {X_train.shape}\nX_test: {X_test.shape}')

In [9]:
def create_sequences(data, seq_len = 8):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len, :])
        y.append(data[i+1:i+seq_len+1, :])
    return np.array(X), np.array(y)

X_train_seq, y_train_seq = create_sequences(X_train)
print(f'{X_train_seq.shape}, {y_train_seq.shape}')

(1003845, 8, 1), (1003845, 8, 1)


In [10]:
from dlfs.layers import DenseLayer, DropoutLayer
from dlfs.activation import Softmax

class SingleAttentionHead():

    def __init__(self, n_embed, head_size, block_size, dropout=0.1):
        self.key = DenseLayer(n_embed, head_size)
        self.query = DenseLayer(n_embed, head_size)
        self.value = DenseLayer(n_embed, head_size)
        self.softmax = Softmax()
        self.dropout = DropoutLayer(dropout)

        self.tril = np.tril(np.ones((block_size, block_size)))
        self.normalize_factor = head_size**0.5

    def forward(self, x, training):
        B, T, C = x.shape

        self.key.forward(x)
        self.query.forward(x)
        self.value.forward(x)

        self.k = self.key.output
        self.q = self.query.output
        self.v = self.value.output
        
        self.w = np.matmul(self.q, self.k.swapaxes(-2, -1)) / self.normalize_factor
        mask_condition = self.tril[:T, :T] == 0
        self.w[:,mask_condition] = -np.inf
        self.softmax.forward(self.w)
        self.w = self.softmax.output
        self.dropout.forward(self.w, training)
        self.w = self.dropout.output
        self.output = np.matmul(self.w, self.v)

    def backward(self, delta):

        # Step 1: Gradient of the loss with respect to w (attention weights)
        d_w = np.matmul(delta, self.v.swapaxes(-2, -1))

        self.dropout.backward(d_w)  # This will apply the dropout mask to the gradients
        d_w = self.dropout.dinputs

        # Step 2: Gradient of the loss with respect to softmax input (logits)
        self.softmax.backward(d_w)  # Softmax backward pass
        d_w = self.softmax.dinputs

        # Step 3: Gradient of the loss with respect to w (before softmax)
        d_w = d_w * (self.w > 0).astype(float)  # Masking out invalid values from softmax

        # Step 4: Gradients w.r.t. key and query using the chain rule
        d_q = np.matmul(d_w, self.k)  # shape: (B, T, head_size)
        d_k = np.matmul(d_w.swapaxes(-2, -1), self.q)  # shape: (B, T, head_size)

        # Step 5: Update the key, query, and value parameters using the gradients
        # Gradient for the key (d_k) and query (d_q) go through the dense layers
        self.key.backward(d_k)
        self.query.backward(d_q)
        self.value.backward(np.matmul(d_w, self.v))

In [11]:
embedding = DenseLayer(1, 192)
head = SingleAttentionHead(n_embed=192, head_size=56, block_size=8)

x = X_train_seq[0]
x = x.reshape(1, *x.shape)
print(f'x {x.shape}')
embedding.forward(x)
print(f'embedding: {embedding.output.shape}')
head.forward(embedding.output, training=True)
print(f'output shape: {head.output.shape}')

x (1, 8, 1)
embedding: (1, 8, 192)
output shape: (1, 8, 56)


In [12]:
delta = np.random.rand(1, 8, 56)
head.backward(delta)
print(head.key.dinputs.shape, head.query.dinputs.shape, head.value.dinputs.shape)

(1, 8, 192) (1, 8, 192) (1, 8, 192)


In [13]:
class MultiHeadAttention():

    def __init__(self, n_embed, n_heads, head_size, block_size, dropout=0.1):
        self.n_heads = n_heads
        self.head_size = head_size

        # List to store each individual attention head
        self.attention_heads = [
            SingleAttentionHead(n_embed, head_size, block_size, dropout)
            for _ in range(n_heads)
        ]

        # Output Dense layer to combine the heads
        self.output_dense = DenseLayer(n_embed, n_embed)

        self.dropout = DropoutLayer(dropout)

    def forward(self, x, training):

        # Store outputs of all attention heads
        head_outputs = []

        for head in self.attention_heads:
            head.forward(x, training)  # Compute attention for this head
            head_outputs.append(head.output)  # Store the output of each head

        # Concatenate the outputs of all heads along the last dimension (features)
        concatenated_output = np.concatenate(np.array(head_outputs), axis=-1) 

        # Pass the concatenated output through the output dense layer
        self.output_dense.forward(concatenated_output)

        self.dropout.forward(self.output_dense.output, training)

        # Final output
        self.output = self.dropout.output

    def backward(self, delta):

        self.dropout.backward(delta)

        self.output_dense.backward(self.dropout.dinputs)

        d_concatenated_output = self.output_dense.output

        # Step 2: Split the gradient back into the individual heads
        d_head_outputs = np.split(d_concatenated_output, self.n_heads, axis=-1)

        # Step 3: Backpropagate through each attention head
        for i, head in enumerate(self.attention_heads):
            head.backward(d_head_outputs[i])  # Backprop through each head


In [14]:
n_embed = 192
n_heads = 8
head_size = n_embed // n_heads

multihead = MultiHeadAttention(n_embed=n_embed, n_heads=n_heads, head_size=head_size, block_size=8)

embedding.forward(x)
multihead.forward(embedding.output, training=True)
print(f'x: {x.shape}')
print(f'embedding: {embedding.output.shape}')
print(f'multihead: {multihead.output.shape}')

x: (1, 8, 1)
embedding: (1, 8, 192)
multihead: (1, 8, 192)


In [15]:
delta = np.random.rand(1, 8, 192)
multihead.backward(delta)
for idx, head in enumerate(multihead.attention_heads):
    print(idx, head.key.dinputs.shape, head.query.dinputs.shape, head.value.dinputs.shape)

0 (1, 8, 192) (1, 8, 192) (1, 8, 192)
1 (1, 8, 192) (1, 8, 192) (1, 8, 192)
2 (1, 8, 192) (1, 8, 192) (1, 8, 192)
3 (1, 8, 192) (1, 8, 192) (1, 8, 192)
4 (1, 8, 192) (1, 8, 192) (1, 8, 192)
5 (1, 8, 192) (1, 8, 192) (1, 8, 192)
6 (1, 8, 192) (1, 8, 192) (1, 8, 192)
7 (1, 8, 192) (1, 8, 192) (1, 8, 192)


In [16]:
from dlfs.activation import ReLU

class FeedForward():

    def __init__(self, n_embed, dropout=0.1):
        self.fc1 = DenseLayer(n_embed, 4*n_embed)
        self.relu1 = ReLU()
        self.fc2 = DenseLayer(4*n_embed, n_embed)
        self.relu2 = ReLU()
        self.dropout = DropoutLayer(dropout)

    def forward(self, inputs, training):
        self.fc1.forward(inputs)
        self.relu1.forward(self.fc1.output)
        self.fc2.forward(self.relu1.output)
        self.relu2.forward(self.fc2.output)
        self.dropout.forward(self.relu2.output, training)
        self.output = self.dropout.output

    def backward(self, delta):
        self.dropout.backward(delta)
        self.relu2.backward(self.dropout.dinputs)
        self.fc2.backward(self.relu2.dinputs)
        self.relu1.backward(self.fc2.dinputs)
        self.fc1.backward(self.relu1.dinputs)
        self.dinputs = self.fc1.dinputs

In [17]:
class LayerNorm:
    def __init__(self, num_features, epsilon=1e-5):
        """
        Initializes the LayerNorm layer.
        
        :param num_features: The number of features in the input (i.e., the dimension to normalize over).
        :param epsilon: Small value to prevent division by zero when computing the standard deviation.
        """
        self.num_features = num_features
        self.epsilon = epsilon
        
        # Initialize the scale (gamma) and shift (beta) parameters
        self.gamma = np.ones(num_features)  # Shape: num_features
        self.beta = np.zeros(num_features)  # Shape: (1, num_features)
        
    def forward(self, inputs):
        """
        Forward pass of LayerNorm
        
        :param x: Input data of shape (batch_size, num_features)
        :return: Layer normalized output
        """
        mean = np.mean(inputs, axis=-1, keepdims=True)
        variance = np.var(inputs, axis=-1, keepdims=True)

        self.normalized = (inputs - mean) / np.sqrt(variance + self.epsilon)
        self.output = self.gamma * self.normalized + self.beta
    
    def backward(self, delta):
        """
        Backward pass for LayerNorm, computing the gradients.
        
        :param dout: The gradient of the loss with respect to the output.
        :return: Gradients with respect to input (dx), gamma, and beta.
        """
        self.dbeta = np.sum(delta, axis=(0, 1))
        self.dgamma = np.sum(delta * self.normalized, axis=(0, 1))
        dnorm = delta * self.gamma
        self.dinputs = dnorm - np.mean(dnorm, axis=-1, keepdims=True) - self.normalized * np.mean(dnorm * self.normalized, axis=-1, keepdims=True)
        

In [18]:
np.random.seed(21)

data = np.random.rand(1, 8, 15)
ln = LayerNorm(15)
ln.forward(data)
print(ln.output.shape)

(1, 8, 15)


In [19]:
delta = np.random.rand(1, 8, 15)
ln.backward(delta)
print(ln.dgamma.shape, ln.dbeta.shape, ln.dinputs.shape)

(15,) (15,) (1, 8, 15)


In [20]:
class Block:
    def __init__(self, n_embed, n_head, block_size):
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_heads=n_head, head_size=head_size, n_embed=n_embed, block_size=block_size)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = LayerNorm(n_embed)
        self.ln2 = LayerNorm(n_embed)
    def forward(self, x, training):
        self.ln1.forward(x)
        self.sa.forward(self.ln1.output, training)
        x = x + self.sa.output
        self.ln2.forward(x)
        self.ffwd.forward(self.ln2.output, training)
        x = x + self.ffwd.output
        self.output = x

    def backward(self, delta):

        dx = delta
        dffwd = dx  # Gradient to pass to the FeedForward layer
        
        self.ffwd.backward(dffwd)

        self.ln2.backward(dx)
        dln2 = self.ln2.dinputs
        
        dsa = dln2  # Gradient to pass to MultiHeadAttention
        
        self.sa.backward(dsa)
        
        self.ln1.backward(dsa)

        self.dinputs = self.ln1.dinputs

In [21]:
n_embed = 192
n_heads = 8

b = Block(n_embed, n_heads, 8)

x = X_train_seq[0]
x = x.reshape(1, *x.shape)
embedding.forward(x)
b.forward(embedding.output, training=True)
print(f'x: {x.shape}')
print(f'embedding: {embedding.output.shape}')
print(f'block: {b.output.shape}')

x: (1, 8, 1)
embedding: (1, 8, 192)
block: (1, 8, 192)


In [24]:
delta = np.random.rand(1, 8, 192)
b.backward(delta)
print(b.dinputs.shape)
embedding.backward(b.dinputs)
print(embedding.dinputs.shape)

(1, 8, 192)
(1, 8, 1)
